In [7]:
import pandas as pd


def clean_hotel_data(df):
    df = df.copy()
    df.columns = df.columns.str.strip()

    # 1. แปลงปี พ.ศ. -> ค.ศ.
    if "year" in df.columns:
        df["year"] = df["year"].apply(lambda x: x - 543 if x > 2500 else x)

    # 2. กรองข้อมูลที่ไม่ต้องการออก
    if "region" in df.columns:
        df = df[~df["region"].str.contains("ทั่วประเทศ", na=False)]
    if "size_estabish" in df.columns:
        df = df[~df["size_estabish"].str.contains("รวม", na=False)]

    # 3. Replace 'region' เป็นภาษาอังกฤษ
    region_map = {
        "กรุงเทพมหานคร": "Bangkok",
        "ภาคกลาง": "Central",
        "ภาคเหนือ": "Northern",
        "ภาคตะวันออกเฉียงเหนือ": "Northeastern",
        "ภาคใต้": "Southern",
        "ภาคตะวันออก": "Eastern",
        "ภาคตะวันตก": "Western",
    }
    if "region" in df.columns:
        df["region"] = df["region"].str.strip().map(region_map).fillna(df["region"])

    # 4. Replace 'size_estabish' เป็นสัญลักษณ์สั้นๆ
    size_symbol_map = {
        "ต่ำกว่า 60 ห้อง": "<60",
        "60-149 ห้อง": "60-149",
        "150 ห้องขึ้นไป": ">150",
    }
    if "size_estabish" in df.columns:
        df["size_estabish"] = (
            df["size_estabish"]
            .str.strip()
            .map(size_symbol_map)
            .fillna(df["size_estabish"])
        )

    # 5. คำนวณค่าจริง (value * 1000)
    # สมมติว่าคอลัมน์ชื่อ 'value' หากชื่ออื่นให้เปลี่ยนชื่อใน [] ครับ
    if "value" in df.columns:
        # แปลงเป็นตัวเลขก่อนเผื่อมีค่าว่างหรือ string หลุดมา
        df["value"] = pd.to_numeric(df["value"], errors="coerce") * 1000

    # 6. ลบคอลัมน์ที่ไม่ได้ใช้
    df = df.drop(columns=["unit", "source"], errors="ignore")

    # 7. สร้างลำดับ Rank (1, 2, 3)
    rank_map = {"<60": 1, "60-149": 2, ">150": 3}
    df["size_rank"] = df["size_estabish"].map(rank_map)

    return df


# --- การโหลดและแสดงผล ---
try:
    df = pd.read_csv("../data/raw/datas.csv", encoding="cp874")
except:
    df = pd.read_csv("../data/raw/datas.csv", encoding="utf-8")

df_cleaned = clean_hotel_data(df)

print("--- ข้อมูล 50 แถวแรก (Value คูณ 1000 แล้ว) ---")
display(df_cleaned.head(50))

--- ข้อมูล 50 แถวแรก (Value คูณ 1000 แล้ว) ---


,﻿year,region,size_estabish,value,size_rank
45,2542,Bangkok,<60,2.970617e+08,1
46,2544,Bangkok,<60,3.161496e+08,1
47,2546,Bangkok,<60,2.924750e+08,1
48,2548,Bangkok,<60,3.187114e+08,1
49,2550,Bangkok,<60,4.439511e+08,1
50,2552,Bangkok,<60,2.247646e+08,1
51,2554,Bangkok,<60,2.701273e+09,1
52,2556,Bangkok,<60,2.158830e+09,1
53,2558,Bangkok,<60,2.785766e+09,1
54,2542,Bangkok,60-149,1.071066e+09,2


In [8]:
# บันทึกเป็น CSV เพื่อส่งต่อให้ไฟล์ modeling
df_cleaned.to_csv("../data/processed/datas_cleaned.csv", index=False)